# **Explicit Classifier**

In this notebook, we leverage on the DenseTact images from our collected dataset to train an explicit classifier that takes in an rgb image of the DenseTact sensor, and outputs the incline angle of the ramp it's in contact, and the material of the ramp.

> **Run record.** These outputs come from a local run on an Apple M4 laptop (PyTorch MPS backend), 2026-11-02, 76 min, status: ok. The printed paths show the local copy of the dataset; in Colab `data_dir` is `/content/drive/MyDrive/tactile_data`.

Let's start by mounting your Google Drive to this notebook. Remember to upload the dataset into your Google Drive Folder! (make sure to connect your runtime to a GPU first)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

[local run] drive.mount('/content/drive') skipped; data_dir points at a local copy


Replace `data_dir` with your own Google Drive directory of the dataset:



In [2]:
data_dir = '/content/drive/MyDrive/tactile_data'

# Dataset
In this section, we will implement DenseTactDataset, which will load tactile images and the correct labels from our directory to the Image classifier.

We start by importing all the necessary libraries, and defining a transformation that we will apply to all images, so that they are more suitable for our network:

In [3]:
import os
import re
import random
import shutil
import cv2
import copy
import pdb
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

# Transforms the images to make them more suitable for ImageNet Structure
dataset_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

### Filtering Out No-Contact Images

The dataset contains images captured throughout the full triangular leg trajectory — including frames where Pupper’s leg is **not in contact** with the surface. However, these no-contact images do not provide useful information about the terrain and may negatively impact our training results. Therefore, we aim to remove them from the dataset.

To identify contact vs. no-contact frames, we use a simple and reliable method that matches the procedure used during data collection. Specifically, we manually sample a small set of reference images known to represent **contact** and another set representing **no-contact**.

For each image in the dataset, we compute the **mean absolute difference (MAD)** between it and both sets of reference images. If the image is more similar (i.e., has a lower MAD) to the contact group than to the no-contact group, we label it as a **contact** frame and keep it. Otherwise, we discard it.

We have provided some relevant helper functions for you:

In [4]:
reference_image_dir = os.path.join(data_dir, "reference_images", "images")
contact_reference_image_paths = [
    os.path.join(reference_image_dir, "image_20250305_005203.jpg"),
    os.path.join(reference_image_dir, "image_20250305_005206.jpg"),
    os.path.join(reference_image_dir, "image_20250305_005207.jpg"),
    os.path.join(reference_image_dir, "image_20250305_005209.jpg"),
    os.path.join(reference_image_dir, "image_20250305_005211.jpg"),
]

no_contact_reference_image_paths = [
    os.path.join(reference_image_dir, "image_20250304_233311.jpg"),
    os.path.join(reference_image_dir, "image_20250304_233312.jpg"),
    os.path.join(reference_image_dir, "image_20250304_233313.jpg"),
    os.path.join(reference_image_dir, "image_20250304_233314.jpg"),
    os.path.join(reference_image_dir, "image_20250304_233315.jpg")
]

def center_crop_and_resize(img, target_size=(224, 224)):
    """
    Crops the center of a 640x480 image to 480x480 and then resizes it to target_size.
    """
    h, w = img.shape[:2]  # Get image dimensions
    crop_size = min(h, w)  # Make it square
    start_x = (w - crop_size) // 2  # Center horizontally
    start_y = (h - crop_size) // 2  # Center vertically

    cropped_img = img[start_y:start_y+crop_size, start_x:start_x+crop_size]
    resized_img = cv2.resize(cropped_img, target_size)
    return resized_img

def image_difference(img1, img2):
    """
    Calculate the mean absolute difference between two images.
    Lower values indicate more similarity.
    """
    if img1.shape != img2.shape:
        img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

    if len(img1.shape) == 3:
        img1_gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    else:
        img1_gray = img1

    if len(img2.shape) == 3:
        img2_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    else:
        img2_gray = img2

    diff = cv2.absdiff(img1_gray, img2_gray)
    return np.mean(diff)

Now, you will help us implement this filtering method below by finishing the `detect_contact` function.

In [5]:
# Reference images get the same center crop and resize as the query image.
# They are read with cv2.imread (BGR channel order), as in the data-collection code.
# With the nearest-reference rule below this keeps exactly 2242 contact frames,
# which matches the 2017 + 225 split sanity check.
contact_images = [center_crop_and_resize(cv2.imread(p)) for p in contact_reference_image_paths]
no_contact_images = [center_crop_and_resize(cv2.imread(p)) for p in no_contact_reference_image_paths]

def detect_contact(image):
    """
    Determines whether the given image corresponds to a contact frame or a no-contact frame.

    This function compares the input image to two sets of reference images: one representing contact,
    and the other representing no-contact. It computes the mean absolute difference (MAD) between
    the input image and each group. If the image is more similar to the contact group
    (i.e., lower MAD), it is classified as a contact image.

    Parameters:
        image (np.ndarray): The input image to classify.

    Returns:
        bool: True if the image is classified as a contact frame, False otherwise.
    """
    image = center_crop_and_resize(image)
    # Distance to a group = MAD to its closest reference image.
    contact_diff = min(image_difference(image, ref) for ref in contact_images)
    no_contact_diff = min(image_difference(image, ref) for ref in no_contact_images)
    return contact_diff < no_contact_diff

### Creating the `DenseTactDataset`

Now that we’ve implemented the contact filtering method, let’s use it to construct the `DenseTactDataset`. This dataset will include only frames where the sensor is in contact with a surface, which helps ensure the model is trained on meaningful terrain information.

**Note:** Loading and filtering all images may take a few minutes.


In [6]:
class DenseTactDataset(Dataset):
    def __init__(self, root_dir, transform=dataset_transform):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []

        # Define class categories
        self.incline_classes = [15, 30, 45, 60, 75, 0]
        self.material_classes = ['none', 'felt pad', 'rough', 'slip', 'very rough', 'foam']

        # Scan dataset structure
        for folder1 in os.listdir(root_dir):
            if folder1[0]=='.' or folder1=="reference_images" or folder1=="lookups":
                continue
            for folder2 in os.listdir(os.path.join(root_dir, folder1)):
                if folder2[0]=='.':
                    continue
                folder_path = None
                if os.path.isdir(os.path.join(root_dir, folder1, folder2)):
                    folder_path = os.path.join(root_dir, folder1, folder2)
                    print(f"Processing {folder_path}")
                if not os.path.isdir(folder_path):
                    continue

                # Parse incline and material from folder name
                incline_label, material_label = self._parse_folder_name(os.path.relpath(folder_path, root_dir))
                print(f"Incline: {self.incline_classes[incline_label]}, Material: {self.material_classes[material_label]}")

                # Skip if we couldn't parse the folder name properly
                if incline_label is None or material_label is None:
                    continue

                # Get images from the folder structure
                self._process_folder(folder_path, incline_label, material_label)

    def _process_folder(self, folder_path, incline_label, material_label):
        """Process a folder to find images and add them to samples"""
        # Check if this is a directory with images directly
        image_dir = None

        # Look for images/ directory first
        if os.path.exists(os.path.join(folder_path, 'images')):
            image_dir = os.path.join(folder_path, 'images')
        else:
            # Check if this folder has subdirectories
            for subfolder in os.listdir(folder_path):
                subfolder_path = os.path.join(folder_path, subfolder)
                if os.path.isdir(subfolder_path):
                    subdir_incline, subdir_material = self._parse_folder_name(subfolder)

                    # If this subfolder has valid classes, update labels and process
                    if subdir_incline is not None:
                        incline_label = subdir_incline
                    if subdir_material is not None:
                        material_label = subdir_material

                    # Look for images directory
                    if os.path.exists(os.path.join(subfolder_path, 'images')):
                        image_dir = os.path.join(subfolder_path, 'images')
                        self._add_images_from_dir(image_dir, incline_label, material_label)

        if image_dir:
            self._add_images_from_dir(image_dir, incline_label, material_label)

    def _add_images_from_dir(self, image_dir, incline_label, material_label):
        """Parallel version: Add all images from a directory to samples"""
        def process_image(img_name):
            if not img_name.endswith(('.jpg', '.jpeg', '.png')):
                return
            img_path = os.path.join(image_dir, img_name)
            image = Image.open(img_path).convert('RGB')
            if detect_contact(np.array(image)):
                self.samples.append((img_path, incline_label, material_label))

        with ThreadPoolExecutor(max_workers=16) as executor:
            executor.map(process_image, os.listdir(image_dir))

    def _parse_folder_name(self, folder_name):
        """Parse folder name to extract incline and material labels"""
        incline_label = None
        material_label = None

        # Convert folder name to lowercase for case-insensitive matching
        folder_lower = folder_name.lower()

        # Try to extract incline
        for i, incline in enumerate(self.incline_classes):
            if str(incline) in folder_lower:
                incline_label = i
                break

        # Try to extract material
        for i, material in enumerate(self.material_classes):
            material_lower = material.lower()
            if material_lower in folder_lower:
                material_label = i
                break

        # Handle special cases
        if 'felt_pad' in folder_lower or 'felt pad' in folder_lower:
            material_label = self.material_classes.index('felt pad')
        elif 'very_rough' in folder_lower or 'very rough' in folder_lower or 'very-rough' in folder_lower:
            material_label = self.material_classes.index('very rough')

        return incline_label, material_label

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, incline_label, material_label = self.samples[idx]

        try:
            image = Image.open(img_path).convert('RGB')

            if self.transform:
                image = self.transform(image)

            return {
                'image': image,
                'incline_label': torch.tensor(incline_label, dtype=torch.long),
                'material_label': torch.tensor(material_label, dtype=torch.long)
            }
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a default or placeholder value
            return None

# Load datasets
train_dataset = DenseTactDataset(data_dir, transform=dataset_transform)

Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/15 degrees_new/15-very_rough
Incline: 15, Material: very rough


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/15 degrees_new/15-rough
Incline: 15, Material: rough


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/15 degrees_new/15-slip
Incline: 15, Material: slip


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/15 degrees_new/15-felt_pad
Incline: 15, Material: felt pad


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees/45-felt_pad
Incline: 45, Material: felt pad


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees/45-very-rough
Incline: 45, Material: very rough


... (40 outputs left out)

Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees_new/45-very_rough
Incline: 45, Material: very rough


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees_new/45-felt_pad
Incline: 45, Material: felt pad


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees_new/45-rough
Incline: 45, Material: rough


Processing /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/45 degrees_new/45-slip
Incline: 45, Material: slip


### Saving Processed Dataset to CSV

Processing these images into a dataset every single time can be slow and inefficient — we definitely don’t want to repeat that each time we train! To avoid this, we’ll save the filtered dataset into a `.csv` file. In the future, we can simply load this preprocessed dataset directly from the CSV!

We’ve provided the CSV saving script below.



In [7]:
# Save dataset to CSV
os.makedirs(os.path.join(data_dir, "lookups"), exist_ok=True)
# The loader threads append samples in arbitrary order; sorting makes the seeded split reproducible.
df = pd.DataFrame(sorted(train_dataset.samples), columns=["image_path", "incline_label", "material_label"])
df.to_csv(os.path.join(data_dir, "lookups/explicit_train_dataset.csv"), index=False)

print(f"Dataset saved successfully at {os.path.join(data_dir, 'lookups/explicit_train_dataset.csv')}!")
print(f"{len(df)} contact images kept")

Dataset saved successfully at /Users/tejkosaraju/Downloads/classes/cs123/tactile_data/lookups/explicit_train_dataset.csv!
2242 contact images kept


Next, you will help us implement the `__getitem__` function for the `DenseTactDatasetFromCSV` class, which will load individual samples from the saved CSV. This is all you need to run in the future to load the dataset!

In [8]:
class DenseTactDatasetFromCSV(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """
        Loads a single sample from the dataset using the image path and labels stored in the CSV file.
        """
        row = self.data.iloc[idx]
        img_path = row['image_path']
        try:
            image = Image.open(img_path).convert('RGB')
        except OSError as e:
            # Fail loudly: returning None would only crash later inside the DataLoader collate.
            raise RuntimeError(f"Could not load image {img_path}") from e

        if self.transform:
            image = self.transform(image)

        return {
            'image': image,
            'incline_label': torch.tensor(int(row['incline_label']), dtype=torch.long),
            'material_label': torch.tensor(int(row['material_label']), dtype=torch.long)
        }


# Load dataset from CSV
train_dataset = DenseTactDatasetFromCSV(os.path.join(data_dir, "lookups/explicit_train_dataset.csv"), transform=dataset_transform)

print("Dataset loaded from CSV successfully!")

Dataset loaded from CSV successfully!


### Splitting the Dataset: Train vs. Validation

Now, let’s split the dataset into **90% training** and **10% validation**.

This step is important because the validation set is used to evaluate the model’s performance on **unseen tactile data**. It does not participate in the training process, helping us understand how well the model generalizes to new inputs.

You will help us use a 9:1 split ratio to divide the dataset accordingly, and construct the DataLoader to feed into the model!

As a sanity check, you should see **2017** train images and **225** validation images after the split.


In [9]:
# Define the split ratio
train_ratio = 0.9
val_ratio = 1 - train_ratio

total_size = len(train_dataset)
train_size = int(train_ratio * total_size)
val_size = total_size - train_size

# Fixed seed so the split (and the reported accuracy) is reproducible.
train_subset, val_subset = random_split(train_dataset, [train_size, val_size],
                                        generator=torch.Generator().manual_seed(0))

dataloaders = {
    'train': DataLoader(train_subset, batch_size=32, shuffle=True),
    'val': DataLoader(val_subset, batch_size=32, shuffle=False)
}

print(f"Dataset split into train ({train_size}) and val ({val_size}) successfully!")

Dataset split into train (2017) and val (225) successfully!


### DenseTactMultiTaskClassifier: A Dual-Head ResNet for Tactile Classification

The `DenseTactMultiTaskClassifier` is a simple dual-headed CNN designed for tactile perception tasks, where a single input image from a tactile sensor is used to simultaneously predict:

- The **incline** of the surface (e.g., flat, sloped, etc.)
- The **material** type (e.g., carpet, metal, wood)

#### 🧠 Architecture Overview

We use a pre-trained **ResNet-50** as the backbone feature extractor. All convolutional layers in the ResNet are unfrozen, so that the model can learn from our own tactile data more effectively.

The original fully connected classification layer is replaced with two **independent heads**, which you will help implement:

- `incline_head`: A linear layer producing logits for 6 incline classes.
- `material_head`: A linear layer producing logits for 6 material classes.

This multi-task design enables the model to share feature representations while making specialized predictions for each task.


In [10]:
# Load pre-trained ResNet50 (ImageNet weights; same as the deprecated pretrained=True)
torch.manual_seed(0)
random.seed(0)
base_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Define the multi-task classifier with separate heads for incline and material
class DenseTactMultiTaskClassifier(nn.Module):
    def __init__(self, base_model, num_incline_classes=6, num_material_classes=6):
        super(DenseTactMultiTaskClassifier, self).__init__()

        # Save base_model and get number of input features to the final layer
        self.base_model = base_model
        num_ftrs = self.base_model.fc.in_features

        # Two independent linear heads on the shared 2048-d ResNet feature
        self.incline_head = nn.Linear(num_ftrs, num_incline_classes)
        self.material_head = nn.Linear(num_ftrs, num_material_classes)

        # Replace the original ResNet classifier with an identity layer
        self.base_model.fc = nn.Identity()

    def forward(self, x):
        # Extract features using the base model
        features = self.base_model(x)

        incline_output = self.incline_head(features)
        material_output = self.material_head(features)

        return {
            "incline": incline_output,
            "material": material_output
        }

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /Users/tejkosaraju/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


### Gradient Surgery (Projected Conflicting Gradient / PCGrad)

PCGrad is a gradient-level technique for **multi-task learning** that helps mitigate conflicts between gradients from different loss functions. It ensures that each task's update direction does not interfere destructively with others.

#### Problem: Conflicting Gradients

In multi-task training, you often have multiple losses:  
$L_1, L_2, \dots, L_K$  
Each loss gives a gradient with respect to model parameters $\theta$:  
$g_i = \nabla_\theta L_i$

These gradients may conflict — for example, if the dot product $\langle g_i, g_j \rangle < 0$, the two tasks are "fighting" during training.

#### PCGrad Solution

1. **Compute individual gradients** for each loss using `retain_graph=True`.

2. **Check for conflict** between gradient pairs:
   - If $\langle g_i, g_j \rangle < 0$, then there is a conflict.

3. **Project $g_i$ to remove the conflicting component** (we do that iteratively on all gradients with randomized order of selecting gradient pairs):
   $$
   g_i \leftarrow g_i - \frac{\langle g_i, g_j \rangle}{\|g_j\|^2} g_j
   $$

4. **Average or sum** the adjusted gradients:
   $$
   g = \frac{1}{K} \sum_{i=1}^{K} g_i
   $$

5. **Set and apply** the final projected gradient with `optimizer.step()`.

#### Why It Matters

PCGrad is especially helpful when:
- Tasks compete (e.g., incline vs. material classification)
- Gradients point in opposing directions
- You want more stable and balanced learning across objectives

Now, you will implement the following methods in the PCGrad class:
- optimizer: Return the wrapped optimizer instance.
- zero_grad: Clear the gradients of all parameters using _optim.zero_grad.
- step: Perform an optimization step using _optim.step.
- pc_backward: Compute gradients for multiple objectives and apply PCGrad projection
- the gradient surgery step in _project_conflicting

using internal methods: _pack_grad, _project_conflicting, _unflatten_grad, and _set_grad.


In [11]:
class PCGrad:
    def __init__(self, optimizer, reduction='mean'):
        """
        Args:
            optimizer (torch.optim.Optimizer): Wrapped optimizer.
            reduction (str): Method to reduce gradients ('mean' or 'sum').
        """
        self._optim = optimizer
        self._reduction = reduction

    @property
    def optimizer(self):
        return self._optim

    def zero_grad(self):
        """Clear gradients of all optimized parameters."""
        return self._optim.zero_grad(set_to_none=True)

    def step(self):
        """Performs a single optimization step."""
        return self._optim.step()

    def pc_backward(self, objectives):
        """
        Compute backward pass using Projected Conflicting Gradient (PCGrad).

        Args:
            objectives (List[torch.Tensor]): List of scalar loss values.
        """
        grads, shapes, has_grads = self._pack_grad(objectives)
        pc_grad = self._project_conflicting(grads, has_grads)
        pc_grad = self._unflatten_grad(pc_grad, shapes[0])
        self._set_grad(pc_grad)

    # ---------- Internal Utilities ----------

    def _pack_grad(self, objectives):
        """
        Compute gradients for each objective.

        Returns:
            grads (List[Tensor]): Flattened gradients.
            shapes (List[Tuple]): Shapes of original parameters.
            has_grads (List[Tensor]): Masks indicating valid gradients.
        """
        grads, shapes, has_grads = [], [], []

        for obj in objectives:
            self._optim.zero_grad(set_to_none=True)
            obj.backward(retain_graph=True)
            grad, shape, has_grad = self._retrieve_grad()
            grads.append(self._flatten_grad(grad))
            has_grads.append(self._flatten_grad(has_grad))
            shapes.append(shape)

        return grads, shapes, has_grads

    def _retrieve_grad(self):
        """
        Retrieve gradients and shapes from the model parameters.

        Returns:
            grad (List[Tensor]): Gradient tensors.
            shape (List[Tuple]): Original parameter shapes.
            has_grad (List[Tensor]): Gradient masks.
        """
        grad, shape, has_grad = [], [], []

        for group in self._optim.param_groups:
            for p in group['params']:
                if p.grad is None:
                    shape.append(p.shape)
                    grad.append(torch.zeros_like(p))
                    has_grad.append(torch.zeros_like(p))
                else:
                    shape.append(p.grad.shape)
                    grad.append(p.grad.clone())
                    has_grad.append(torch.ones_like(p))

        return grad, shape, has_grad

    def _flatten_grad(self, grads):
        return torch.cat([g.flatten() for g in grads])

    def _unflatten_grad(self, flat_grads, shapes):
        unflattened, idx = [], 0
        for shape in shapes:
            length = np.prod(shape)
            unflattened.append(flat_grads[idx:idx + length].view(shape).clone())
            idx += length
        return unflattened

    def _set_grad(self, grads):
        """Assign updated gradients back to model parameters."""
        idx = 0
        for group in self._optim.param_groups:
            for p in group['params']:
                p.grad = grads[idx]
                idx += 1

    def _project_conflicting(self, grads, has_grads):
        """
        Resolve gradient conflicts via orthogonal projection.

        Args:
            grads (List[Tensor]): Flattened gradients.
            has_grads (List[Tensor]): Valid gradient masks.

        Returns:
            Tensor: Projected merged gradient.
        """
        shared = torch.stack(has_grads).prod(0).bool()
        pc_grad = copy.deepcopy(grads)
        num_tasks = len(grads)

        for g_i in pc_grad:
            random.shuffle(grads)
            for g_j in grads:
                # Gradient surgery: if g_i conflicts with g_j (negative inner product),
                # remove from g_i its component along g_j (in place, so pc_grad is updated).
                g_i_g_j = torch.dot(g_i, g_j)
                if g_i_g_j < 0:
                    g_i -= g_i_g_j * g_j / (g_j.norm() ** 2)

        merged = torch.zeros_like(grads[0])

        # Handle weights with all contributing gradients
        if self._reduction == 'mean':
            merged[shared] = torch.stack([g[shared] for g in pc_grad]).mean(dim=0)
        elif self._reduction == 'sum':
            merged[shared] = torch.stack([g[shared] for g in pc_grad]).sum(dim=0)
        else:
            raise ValueError("Invalid reduction method")
        # Sum up weights with some non-contributing gradients for stability
        merged[~shared] = torch.stack([g[~shared] for g in pc_grad]).sum(dim=0)
        return merged

### Training the Multi-Class Tactile Classifier with LoRA and PCGrad (no implementation needed)

In this section, we train a multi-output classifier on tactile sensor images, using two distinct classification heads:

- **Incline classification**: predicting the slope of the surface the robot is contacting.
- **Material classification**: identifying the type of terrain material (e.g., carpet, wood, metal, etc.).

#### ⚙️ Model Architecture
We use a pre-trained **ResNet-50** backbone as the feature extractor. Its final classification layer is replaced by **two parallel heads**, one for each task. This setup allows the model to share low-level vision features while learning to make **independent predictions** for incline and material.

#### 🪶 LoRA for Efficient Fine-Tuning
We optionally apply **LoRA (Low-Rank Adaptation)** to just the two output heads. This allows for **parameter-efficient fine-tuning** by injecting trainable low-rank matrices into the linear layers of interest. All other layers in the pre-trained ResNet remain frozen, saving compute and reducing the risk of overfitting.

#### 🔀 PCGrad: Handling Multi-Objective Conflicts
Because we’re optimizing two loss functions simultaneously (one for each head), **PCGrad (Projected Conflicting Gradient)** is used. PCGrad modifies gradients during training to resolve conflicts between competing objectives, improving training stability in **multi-task learning** settings.

#### 🔄 Training and Validation
We train the model over 10 epochs using:
- **CrossEntropyLoss** for both incline and material labels
- **AdamW optimizer** wrapped by PCGrad
- A 90/10 train-validation split

After each epoch, we report:
- Total loss
- Accuracy for incline classification
- Accuracy for material classification

The validation set helps evaluate model generalization on unseen tactile data.



In [12]:
# Optional: intall peft, which enables LoRA on training
%pip install -q peft
from peft import LoraConfig, get_peft_model

Note: you may need to restart the kernel to use updated packages.


/private/tmp/claude-501/-Users-tejkosaraju-Downloads-classes-cs123/15934c31-ca98-4461-a1c3-e3b9024ec06a/scratchpad/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1102 19:03:22.260000 21561 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [13]:
# --- Model Definition (Already Defined Above) ---
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = DenseTactMultiTaskClassifier(base_model).to(device)

# Optional: Enable LoRA for classification heads only
# (Commented for student exercise or runtime flexibility)
'''
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["incline_head", "material_head"],
    lora_dropout=0.1,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
'''

# --- Loss Functions & Optimizer ---
criterion_incline = nn.CrossEntropyLoss()
criterion_material = nn.CrossEntropyLoss()

optimizer = optim.AdamW(model.parameters(), lr=0.001)
optimizer = PCGrad(optimizer)  # Multi-objective gradient conflict handling

# --- Training Loop ---
num_epochs = 10
for epoch in range(num_epochs):
    print(f'Epoch {epoch + 1}/{num_epochs}\n{"-" * 10}')

    for phase in ['train', 'val']:
        model.train() if phase == 'train' else model.eval()

        running_loss = 0.0
        correct_incline = 0
        correct_material = 0

        for batch in dataloaders[phase]:
            images = batch['image'].to(device)
            incline_labels = batch['incline_label'].to(device)
            material_labels = batch['material_label'].to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(images)
                loss_incline = criterion_incline(outputs['incline'], incline_labels)
                loss_material = criterion_material(outputs['material'], material_labels)
                total_loss = loss_incline + loss_material

                if phase == 'train':
                    optimizer.pc_backward([loss_incline, loss_material])
                    optimizer.step()

            # Prediction and accuracy tracking
            incline_preds = torch.argmax(torch.softmax(outputs['incline'], dim=1), dim=1)
            material_preds = torch.argmax(torch.softmax(outputs['material'], dim=1), dim=1)

            running_loss += total_loss.item() * images.size(0)
            correct_incline += (incline_preds == incline_labels).sum().item()
            correct_material += (material_preds == material_labels).sum().item()

        epoch_loss = running_loss / len(dataloaders[phase].dataset)
        epoch_acc_incline = correct_incline / len(dataloaders[phase].dataset)
        epoch_acc_material = correct_material / len(dataloaders[phase].dataset)

        print(f'{phase} | Loss: {epoch_loss:.4f} | Incline Acc: {epoch_acc_incline:.4f} | Material Acc: {epoch_acc_material:.4f}')

print("Training complete!")

Epoch 1/10
----------


train | Loss: 1.7995 | Incline Acc: 0.7769 | Material Acc: 0.4958


val | Loss: 10.2785 | Incline Acc: 0.3956 | Material Acc: 0.2400
Epoch 2/10
----------


train | Loss: 1.0906 | Incline Acc: 0.9048 | Material Acc: 0.6807


val | Loss: 1.7765 | Incline Acc: 0.6711 | Material Acc: 0.7289
Epoch 3/10
----------


train | Loss: 0.7513 | Incline Acc: 0.9405 | Material Acc: 0.7838


... (11 outputs left out)

train | Loss: 0.2587 | Incline Acc: 0.9787 | Material Acc: 0.9286


val | Loss: 1.4052 | Incline Acc: 0.9689 | Material Acc: 0.7244
Epoch 10/10
----------


train | Loss: 0.2880 | Incline Acc: 0.9797 | Material Acc: 0.9271


val | Loss: 0.4974 | Incline Acc: 0.9822 | Material Acc: 0.8978
Training complete!


### Results (added)

Final-epoch validation accuracy, per-class accuracy, and confusion matrices on the 225 held-out images.
Rows are the true class and columns are the predicted class.

In [14]:
model.eval()
inc_true, inc_pred, mat_true, mat_pred = [], [], [], []
with torch.no_grad():
    for batch in dataloaders['val']:
        out = model(batch['image'].to(device))
        inc_pred += out['incline'].argmax(1).cpu().tolist()
        mat_pred += out['material'].argmax(1).cpu().tolist()
        inc_true += batch['incline_label'].tolist()
        mat_true += batch['material_label'].tolist()
inc_true, inc_pred, mat_true, mat_pred = map(np.array, (inc_true, inc_pred, mat_true, mat_pred))

print(f"Validation incline accuracy:  {(inc_true == inc_pred).mean():.4f}")
print(f"Validation material accuracy: {(mat_true == mat_pred).mean():.4f}")
print(f"Both correct:                 {((inc_true == inc_pred) & (mat_true == mat_pred)).mean():.4f}")

incline_names = [f"{a} deg" for a in [15, 30, 45, 60, 75, 0]]
material_names = ['none', 'felt pad', 'rough', 'slip', 'very rough', 'foam']
for name, t, p, labels in (("Incline", inc_true, inc_pred, incline_names), ("Material", mat_true, mat_pred, material_names)):
    cm = pd.crosstab(pd.Series([labels[i] for i in t], name='true'), pd.Series([labels[i] for i in p], name='pred'))
    print(f"\n{name} confusion matrix (validation)")
    print(cm.to_string())

Validation incline accuracy:  0.9822
Validation material accuracy: 0.8978
Both correct:                 0.8889

Incline confusion matrix (validation)
pred    0 deg  15 deg  30 deg  45 deg  60 deg  75 deg
true                                                 
0 deg      31       1       1       0       0       0
15 deg      0      46       1       0       0       0
30 deg      0       0      51       0       0       0
45 deg      0       0       0      42       0       0
60 deg      0       0       0       0      32       1
75 deg      0       0       0       0       0      19

Material confusion matrix (validation)
pred        felt pad  foam  none  rough  slip  very rough
true                                                     
felt pad          38     0     0      1     0           4
foam               1    31     0      0     0           3
none               0     0    15      0     1           0
rough              1     0     0     27     0           6
slip               0     0    

Bravo! You finished training an explicit classifier that detects terrain slope and material given the image reading of the DenseTact sensor. You should get around **90%** on the validation accuracy on both the slope detection and material detection, proving that fine-grained tactile sensors like DenseTact are powerful enough for this task!

However, you may also notice that the validation accuracy is not very stable-a sign that our collected data (~2400 images) aren't enough to fully train the image classifier. We will see if that is the case for training the implicit classifier.